In [26]:
"""
FAST, visible, cloud-native HLS + Prithvi pipeline

What this version is designed to do:
1. Search HLS metadata from the cloud.
2. Open only a small number of candidate cloud COGs.
3. Never use earthaccess.download().
4. Never save big local image stacks unless SAVE_LOCAL_STACKS=True.
5. Always save visible outputs:
   - final summary CSV
   - partial summary CSV
   - per-state/season candidate diagnostics CSVs
   - per-state/season HLS index statistics CSVs when successful
   - RGB preview PNGs when a usable chip is created

Important:
This extracts features and vegetation statistics. It is NOT a trained yield predictor yet.
"""

# ============================================================
# 1. Imports
# ============================================================

import os
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
import torch
import torch.nn as nn
import earthaccess
import matplotlib.pyplot as plt
from shapely.geometry import box

from terratorch.registry import BACKBONE_REGISTRY


# ============================================================
# 2. User settings
# ============================================================

DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "outputs"
INDEX_DIR = DATA_DIR / "index_outputs"
DIAGNOSTIC_DIR = DATA_DIR / "diagnostics"
PREVIEW_DIR = DATA_DIR / "preview_images"
STACK_DIR = DATA_DIR / "model_stacks"

SAVE_LOCAL_STACKS = False
RUN_PRITHVI = True
ALLOW_PARTIAL_FORECAST_DATES = True
PROCESS_STATES = ["Colorado", "Iowa", "Missouri", "Nebraska", "Wisconsin"]

MAX_SEARCH_RESULTS_PER_WINDOW = 100
MAX_CANDIDATE_GRANULES_TO_OPEN = 12
STOP_AFTER_FIRST_USABLE_CHIP = True

MAX_CLOUDY_PIXEL_PERCENT = 80.0
MIN_VALID_PIXEL_PERCENT = 5.0

TARGET_SIZE = 224
PIXEL_SIZE_METERS = 30
CHIP_SIZE_METERS = TARGET_SIZE * PIXEL_SIZE_METERS
HALF_CHIP_METERS = CHIP_SIZE_METERS / 2
CHUNK_SIZE = dict(band=1, x=512, y=512)

HLS_SHORT_NAMES = ["HLSL30", "HLSS30"]

BOUNDARY_FILES = {
    "Colorado": "Field_Boundary_Colorado.geojson",
    "Iowa": "Field_Boundary_Iowa.geojson",
    "Missouri": "Field_Boundary_Missouri.geojson",
    "Nebraska": "Field_Boundary_Nebraska.geojson",
    "Wisconsin": "Field_Boundary_Wisconsin.geojson",
}

# The hackathon prompt asks for four specific forecast points:
#   August 1, September 1, October 1, and Final / End of Season.
# It also says the deliverable should be for the 2025 season.
FORECAST_YEAR = 2025

# HLS can miss an exact date because of cloud cover or satellite revisit timing.
# Use EXACT_DATE_WINDOW_DAYS = 2 for literally only that date.
# Use 1 or 2 if the exact date has no usable imagery.
EXACT_DATE_WINDOW_DAYS = 16

END_OF_SEASON_DATES = {
    "Iowa": "10-10",
    "Colorado": "10-20",
    "Missouri": "09-30",
    "Wisconsin": "10-20",
    "Nebraska": "10-05",
}

STATE_DATES = {
    state: {
        "august_1": f"{FORECAST_YEAR}-08-01",
        "september_1": f"{FORECAST_YEAR}-09-01",
        "october_1": f"{FORECAST_YEAR}-10-01",
        "end_of_season": f"{FORECAST_YEAR}-{END_OF_SEASON_DATES[state]}",
    }
    for state in END_OF_SEASON_DATES
}

DATE_POINT_ORDER = ["august_1", "september_1", "october_1", "end_of_season"]


def make_temporal_window(date_string: str, pad_days: int = EXACT_DATE_WINDOW_DAYS) -> Tuple[str, str]:
    """
    Turns a specific forecast date into an HLS temporal search window.

    With pad_days=0:
        2025-08-01 becomes 2025-08-01T00:00:00 to 2025-08-01T23:59:59

    With pad_days=1:
        searches one day before through one day after.
    """
    center = pd.to_datetime(date_string)
    start = center - pd.Timedelta(days=pad_days)
    end = center + pd.Timedelta(days=pad_days)
    return (
        start.strftime("%Y-%m-%dT00:00:00"),
        end.strftime("%Y-%m-%dT23:59:59"),
    )
MODEL_BAND_ORDER = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]

HLS_BAND_MAP = {
    "HLSS30": {"Blue": "B02", "Green": "B03", "Red": "B04", "NIR": "B8A", "SWIR1": "B11", "SWIR2": "B12", "Fmask": "Fmask"},
    "HLSL30": {"Blue": "B02", "Green": "B03", "Red": "B04", "NIR": "B05", "SWIR1": "B06", "SWIR2": "B07", "Fmask": "Fmask"},
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")

for folder in [DATA_DIR, OUTPUT_DIR, INDEX_DIR, DIAGNOSTIC_DIR, PREVIEW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if SAVE_LOCAL_STACKS:
    STACK_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. Setup helpers
# ============================================================

def configure_cloud_cog_access() -> None:
    cookie_file_path = os.path.expanduser("~/cookies.txt")
    os.environ["GDAL_HTTP_COOKIEFILE"] = cookie_file_path
    os.environ["GDAL_HTTP_COOKIEJAR"] = cookie_file_path
    os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
    os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = "TIF"
    os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"
    os.environ["GDAL_HTTP_MAX_RETRY"] = "10"
    os.environ["GDAL_HTTP_RETRY_DELAY"] = "0.5"


def safe_scalar(value) -> float:
    """Convert xarray/dask/numpy/python scalar to a plain float."""
    try:
        if hasattr(value, "compute"):
            value = value.compute()
        if hasattr(value, "item"):
            value = value.item()
        return float(value)
    except Exception:
        return float("nan")


# ============================================================
# 4. Boundary and ROI helpers
# ============================================================

def check_boundary_files() -> None:
    missing = []
    for state in PROCESS_STATES:
        path = DATA_DIR / BOUNDARY_FILES[state]
        if not path.exists():
            missing.append(path)
    if missing:
        for path in missing:
            print(f"Missing: {path}")
        raise FileNotFoundError("Missing one or more boundary GeoJSON files.")


def load_boundary(state_name: str) -> gpd.GeoDataFrame:
    path = DATA_DIR / BOUNDARY_FILES[state_name]
    gdf = gpd.read_file(path)
    if gdf.empty:
        raise ValueError(f"Boundary is empty: {path}")
    if gdf.crs is None:
        print(f"WARNING: {state_name} boundary has no CRS. Assuming EPSG:4326.")
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf


def get_bbox(gdf: gpd.GeoDataFrame) -> Tuple[float, float, float, float]:
    minx, miny, maxx, maxy = gdf.total_bounds
    return float(minx), float(miny), float(maxx), float(maxy)


def make_small_roi_from_boundary(boundary_wgs84: gpd.GeoDataFrame, state_name: str) -> gpd.GeoDataFrame:
    boundary_5070 = boundary_wgs84.to_crs("EPSG:5070")
    point = boundary_5070.union_all().representative_point()
    x, y = point.x, point.y
    roi_geom = box(x - HALF_CHIP_METERS, y - HALF_CHIP_METERS, x + HALF_CHIP_METERS, y + HALF_CHIP_METERS)
    roi_5070 = gpd.GeoDataFrame({"state": [state_name]}, geometry=[roi_geom], crs="EPSG:5070")
    return roi_5070.to_crs("EPSG:4326")


# ============================================================
# 5. Metadata helpers
# ============================================================

def get_granule_cloud_cover(granule) -> float:
    try:
        umm = granule.get("umm", {})
        for key in ["CloudCover", "CloudCoverPercentage"]:
            value = umm.get(key)
            if value is not None:
                return float(value)
        for attr in umm.get("AdditionalAttributes", []):
            name = str(attr.get("Name", "")).lower()
            if "cloud" in name:
                values = attr.get("Values", [])
                if values:
                    return float(values[0])
    except Exception:
        pass
    return 9999.0


def get_granule_start_time(granule) -> str:
    try:
        return granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    except Exception:
        return "unknown"


def get_product_type_from_url(url: str) -> Optional[str]:
    if "HLS.S30" in url or "/HLSS30" in url:
        return "HLSS30"
    if "HLS.L30" in url or "/HLSL30" in url:
        return "HLSL30"
    return None


def get_band_from_url(url: str) -> str:
    return url.split("/")[-1].split(".")[-2]


def granule_to_band_links(granule) -> Optional[Dict[str, str]]:
    links = granule.data_links()
    if not links:
        return None
    product_type = get_product_type_from_url(links[0])
    if product_type not in HLS_BAND_MAP:
        return None
    reverse_map = {band_id: common_name for common_name, band_id in HLS_BAND_MAP[product_type].items()}
    band_links = {
        "product_type": product_type,
        "granule_start_time": get_granule_start_time(granule),
        "metadata_cloud_cover": get_granule_cloud_cover(granule),
    }
    for url in links:
        band_id = get_band_from_url(url)
        if band_id in reverse_map:
            band_links[reverse_map[band_id]] = url
    required = MODEL_BAND_ORDER + ["Fmask"]
    if not all(name in band_links for name in required):
        return None
    return band_links


def search_hls_metadata(bbox: Tuple[float, float, float, float], temporal: Tuple[str, str]):
    results = []
    for short_name in HLS_SHORT_NAMES:
        try:
            found = earthaccess.search_data(
                short_name=short_name,
                bounding_box=bbox,
                temporal=temporal,
                count=MAX_SEARCH_RESULTS_PER_WINDOW,
            )
            results.extend(found)
        except Exception as e:
            print(f"    Search failed for {short_name}: {e}")
    print(f"    Total granules found (HLSL30 + HLSS30): {len(results)}")
    return sorted(results, key=get_granule_cloud_cover)


# ============================================================
# 6. Raster helpers
# ============================================================

def open_band(url: str):
    return rxr.open_rasterio(url, chunks=CHUNK_SIZE, masked=True).squeeze("band", drop=True)


def scale_reflectance(da):
    out = da.copy()
    out.data = da.data * 0.0001
    out.attrs["scale_factor"] = 1
    return out


def create_quality_mask(fmask_data, bit_nums: List[int] = [1, 2, 3, 4, 5]) -> np.ndarray:
    quality_data = np.array(fmask_data, dtype=np.float32)
    quality_data = np.where(np.isfinite(quality_data), quality_data, 255).astype(np.uint8)
    mask = np.zeros(quality_data.shape, dtype=bool)
    for bit in bit_nums:
        mask = np.logical_or(mask, (quality_data & (1 << bit)) > 0)
    return mask


def calc_ndvi(red, nir):
    ndvi = red.copy()
    ndvi.data = (nir.data - red.data) / (nir.data + red.data)
    return xr.where(np.isfinite(ndvi), ndvi, np.nan, keep_attrs=True)


def calc_evi(red, blue, nir):
    evi = red.copy()
    evi.data = 2.5 * ((nir.data - red.data) / (nir.data + 6.0 * red.data - 7.5 * blue.data + 1.0))
    return xr.where(np.isfinite(evi), evi, np.nan, keep_attrs=True)


def resize_or_pad(arr: np.ndarray, target_size: int = TARGET_SIZE) -> np.ndarray:
    out = np.zeros((target_size, target_size), dtype=np.float32)
    h = min(arr.shape[0], target_size)
    w = min(arr.shape[1], target_size)
    out[:h, :w] = arr[:h, :w]
    return out


def save_chip_preview(chip: np.ndarray, state_name: str, date_label: str) -> str:
    blue, green, red = chip[0], chip[1], chip[2]
    rgb = np.stack([red, green, blue], axis=-1)
    rgb = np.clip(rgb * 3.0, 0, 1)
    out_path = PREVIEW_DIR / f"{state_name}_{date_label}_rgb_preview.png"
    plt.figure(figsize=(5, 5))
    plt.imshow(rgb)
    plt.axis("off")
    plt.title(f"{state_name} {date_label} HLS RGB")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    return str(out_path)


# ============================================================
# 7. Candidate opening
# ============================================================

def make_skip_stats(state_name, date_label, forecast_date, candidate_index, reason, band_links=None, extra=None):
    row = {
        "state": state_name,
        "date_label": date_label,
        "forecast_date": forecast_date,
        "candidate_index": candidate_index,
        "status": "skipped",
        "skip_reason": reason,
    }
    if band_links:
        row.update({
            "product_type": band_links.get("product_type"),
            "granule_start_time": band_links.get("granule_start_time"),
            "metadata_cloud_cover": band_links.get("metadata_cloud_cover"),
        })
    if extra:
        row.update(extra)
    return {"chip": None, "stats": row}


def open_candidate_to_chip(granule, roi_wgs84: gpd.GeoDataFrame, state_name: str, date_label: str, forecast_date: str, candidate_index: int) -> dict:
    band_links = granule_to_band_links(granule)
    if band_links is None:
        return make_skip_stats(state_name, date_label, forecast_date, candidate_index, "missing required band links")

    try:
        nir_raw = open_band(band_links["NIR"])
        tile_crs = nir_raw.rio.crs
        if tile_crs is None:
            return make_skip_stats(state_name, date_label, forecast_date, candidate_index, "missing tile CRS", band_links)

        roi_tile_crs = roi_wgs84.to_crs(tile_crs)

        fmask = open_band(band_links["Fmask"])
        fmask_cropped = fmask.rio.clip(roi_tile_crs.geometry.values, roi_tile_crs.crs, all_touched=True)
        bad_mask = create_quality_mask(fmask_cropped.data)

        cloudy_pct = float(np.mean(bad_mask) * 100.0)
        valid_pct = 100.0 - cloudy_pct

        if cloudy_pct > MAX_CLOUDY_PIXEL_PERCENT:
            return make_skip_stats(
                state_name, date_label, forecast_date, candidate_index,
                f"too cloudy after ROI clip ({cloudy_pct:.1f}%)",
                band_links,
                {"cloudy_pixel_pct_after_clip": round(cloudy_pct, 2), "valid_pixel_pct_after_clip": round(valid_pct, 2)},
            )

        if valid_pct < MIN_VALID_PIXEL_PERCENT:
            return make_skip_stats(
                state_name, date_label, forecast_date, candidate_index,
                f"not enough valid pixels ({valid_pct:.1f}%)",
                band_links,
                {"cloudy_pixel_pct_after_clip": round(cloudy_pct, 2), "valid_pixel_pct_after_clip": round(valid_pct, 2)},
            )

        raw_bands = {}
        for band_name in MODEL_BAND_ORDER:
            da = nir_raw if band_name == "NIR" else open_band(band_links[band_name])
            clipped = da.rio.clip(roi_tile_crs.geometry.values, roi_tile_crs.crs, all_touched=True)
            raw_bands[band_name] = scale_reflectance(clipped)

        ndvi_qf = calc_ndvi(raw_bands["Red"], raw_bands["NIR"]).where(~bad_mask)
        evi_qf = calc_evi(raw_bands["Red"], raw_bands["Blue"], raw_bands["NIR"]).where(~bad_mask)

        mean_ndvi = safe_scalar(ndvi_qf.mean(skipna=True))
        mean_evi = safe_scalar(evi_qf.mean(skipna=True))

        chip_bands = []
        for band_name in MODEL_BAND_ORDER:
            band_qf = raw_bands[band_name].where(~bad_mask)
            arr = np.array(band_qf.data, dtype=np.float32)
            arr = np.where(np.isfinite(arr), arr, 0.0).astype(np.float32)
            arr = np.clip(arr, 0.0, 1.0)
            chip_bands.append(resize_or_pad(arr))

        chip = np.stack(chip_bands, axis=0).astype(np.float32)
        preview_path = save_chip_preview(chip, state_name, date_label)

        stats = {
            "state": state_name,
            "date_label": date_label,
            "forecast_date": forecast_date,
            "candidate_index": candidate_index,
            "status": "usable",
            "skip_reason": "",
            "product_type": band_links["product_type"],
            "granule_start_time": band_links["granule_start_time"],
            "metadata_cloud_cover": band_links["metadata_cloud_cover"],
            "cloudy_pixel_pct_after_clip": round(cloudy_pct, 2),
            "valid_pixel_pct_after_clip": round(valid_pct, 2),
            "mean_ndvi": mean_ndvi,
            "median_ndvi": mean_ndvi,
            "mean_evi": mean_evi,
            "median_evi": mean_evi,
            "chip_shape": str(tuple(chip.shape)),
            "preview_path": preview_path,
        }
        return {"chip": chip, "stats": stats}

    except Exception as exc:
        return make_skip_stats(state_name, date_label, forecast_date, candidate_index, str(exc), band_links)


# ============================================================
# 8. Season processing
# ============================================================

def process_date_point(state_name: str, date_label: str, forecast_date: str, roi_wgs84: gpd.GeoDataFrame, roi_bbox: Tuple[float, float, float, float]) -> dict:
    temporal = make_temporal_window(forecast_date)
    print(f" Forecast point: {date_label}")
    print(f"  Specific date: {forecast_date}")
    print(f"  HLS search window: {temporal[0]} to {temporal[1]}")

    granules = search_hls_metadata(roi_bbox, temporal)
    print(f"  Metadata granules found for small ROI: {len(granules)}")

    candidates_to_open = granules[:MAX_CANDIDATE_GRANULES_TO_OPEN]
    print(f"  Opening best candidates only: {len(candidates_to_open)}")

    usable_candidates = []
    diagnostic_rows = []

    for idx, granule in enumerate(candidates_to_open, start=1):
        cloud = get_granule_cloud_cover(granule)
        cloud_text = "unknown" if cloud == 9999.0 else f"{cloud:.1f}%"
        print(f"    Candidate {idx}/{len(candidates_to_open)} | metadata cloud={cloud_text}")

        result = open_candidate_to_chip(granule, roi_wgs84, state_name, date_label, forecast_date, idx)
        diagnostic_rows.append(result["stats"])

        if result["chip"] is None:
            print(f"      skipped: {result['stats'].get('skip_reason')}")
            continue

        usable_candidates.append(result)
        print(
            "      usable | "
            f"valid={result['stats']['valid_pixel_pct_after_clip']}% | "
            f"cloudy={result['stats']['cloudy_pixel_pct_after_clip']}% | "
            f"mean EVI={result['stats']['mean_evi']:.4f} | "
            f"preview={result['stats']['preview_path']}"
        )

        if STOP_AFTER_FIRST_USABLE_CHIP:
            break

    diag_path = DIAGNOSTIC_DIR / f"{state_name}_{date_label}_candidate_diagnostics.csv"
    pd.DataFrame(diagnostic_rows).to_csv(diag_path, index=False)
    print(f"  Candidate diagnostics saved: {diag_path}")

    if not usable_candidates:
        raise RuntimeError(f"No usable chip for {state_name} {date_label} ({forecast_date}). See diagnostics: {diag_path}")

    usable_candidates = sorted(
        usable_candidates,
        key=lambda x: (x["stats"].get("valid_pixel_pct_after_clip", 0), -x["stats"].get("cloudy_pixel_pct_after_clip", 999)),
        reverse=True,
    )
    return usable_candidates[0]


# ============================================================
# 9. Prithvi helpers
# ============================================================

def build_temporal_stack(chips: List[np.ndarray]) -> np.ndarray:
    return np.stack(chips, axis=0).astype(np.float32)


def make_model_inputs(stack: np.ndarray) -> Dict[str, torch.Tensor]:
    channel_first_5d = torch.from_numpy(
        stack.transpose(1, 0, 2, 3).copy()  # .copy() prevents non-contiguous array errors
    ).unsqueeze(0).float().to(DEVICE)        # .float() and .to(DEVICE) are critical
    return {"channel_first_5d": channel_first_5d}


def run_prithvi(model: nn.Module, stack: np.ndarray):
    # Move all parameters and buffers to DEVICE
    model.to(DEVICE)
    for name, buf in model.named_buffers():
        buf.data = buf.data.to(DEVICE)

    inputs = make_model_inputs(stack)
    errors = {}
    for input_name, tensor in inputs.items():
        try:
            print(f"  Trying Prithvi input {input_name}: {tuple(tensor.shape)}")
            print(f"  Tensor device: {tensor.device}")
            # Patch any internal torch.zeros/ones/arange calls to default to DEVICE
            with torch.no_grad():
                with torch.cuda.device(DEVICE):
                    features = model(tensor)
            return features, input_name
        except Exception as exc:
            errors[input_name] = str(exc)

    # If CUDA keeps failing, fall back to CPU entirely
    print("  WARNING: CUDA path failed, attempting CPU fallback...")
    model.to("cpu")
    try:
        stack_cpu = torch.from_numpy(
            stack.transpose(1, 0, 2, 3).copy()
        ).unsqueeze(0).float().cpu()
        with torch.no_grad():
            features = model(stack_cpu)
        model.to(DEVICE)  # move back after
        return features, "cpu_fallback"
    except Exception as exc:
        errors["cpu_fallback"] = str(exc)

    raise RuntimeError(f"Prithvi failed for all input shapes: {errors}")


def summarize_features(features) -> dict:
    if isinstance(features, dict):
        tensors = [v for v in features.values() if torch.is_tensor(v)]
        if tensors:
            features = tensors[-1]
    if isinstance(features, (list, tuple)):
        tensors = [v for v in features if torch.is_tensor(v)]
        if tensors:
            features = tensors[-1]
    if not torch.is_tensor(features):
        raise ValueError(f"Unsupported Prithvi output type: {type(features)}")
    f = features.detach().float().cpu()
    return {
        "feature_shape": str(tuple(f.shape)),
        "feature_mean": round(float(f.mean().item()), 6),
        "feature_std": round(float(f.std().item()), 6),
        "feature_min": round(float(f.min().item()), 6),
        "feature_max": round(float(f.max().item()), 6),
    }


# ============================================================
# 10. State processing
# ============================================================

def process_state(state_name: str, model: Optional[nn.Module]) -> dict:
    print("\n" + "=" * 70)
    print(f"Processing {state_name}")
    print("=" * 70)

    state_boundary = load_boundary(state_name)
    state_bbox = get_bbox(state_boundary)
    roi_wgs84 = make_small_roi_from_boundary(state_boundary, state_name)
    roi_bbox = get_bbox(roi_wgs84)

    print(f"State bbox: {state_bbox}")
    print(f"Small ROI bbox used for HLS search: {roi_bbox}")
    print(f"Small ROI is about {CHIP_SIZE_METERS / 1000:.2f} km x {CHIP_SIZE_METERS / 1000:.2f} km")

    chips = []
    rows = []

    for date_label in DATE_POINT_ORDER:
        forecast_date = STATE_DATES[state_name][date_label]
        try:
            result = process_date_point(state_name, date_label, forecast_date, roi_wgs84, roi_bbox)
            chips.append(result["chip"])
            rows.append(result["stats"])
        except Exception as exc:
            print(f"  WARNING: {state_name} {date_label} failed: {exc}")
            rows.append({
                "state": state_name,
                "date_label": date_label,
                "forecast_date": forecast_date,
                "status": "failed",
                "skip_reason": str(exc),
                "mean_ndvi": np.nan,
                "median_ndvi": np.nan,
                "mean_evi": np.nan,
                "median_evi": np.nan,
                "cloudy_pixel_pct_after_clip": np.nan,
                "valid_pixel_pct_after_clip": np.nan,
                "preview_path": "",
            })
            if not ALLOW_PARTIAL_FORECAST_DATES:
                raise

    season_df = pd.DataFrame(rows)
    season_stats_path = INDEX_DIR / f"{state_name}_forecast_date_hls_stats.csv"
    season_df.to_csv(season_stats_path, index=False)
    print(f"  Season stats saved: {season_stats_path}")

    if len(chips) == 0:
        raise RuntimeError(f"No usable chips at any forecast date for {state_name}")

    # Prithvi stack only includes dates that successfully produced usable chips.
    # The CSV still records failed dates so you can see exactly what happened.
    stack = build_temporal_stack(chips)
    print(f"  In-memory model stack shape from usable dates only: {stack.shape}")

    stack_path = "not_saved_cloud_streamed_in_memory"
    if SAVE_LOCAL_STACKS:
        save_path = STACK_DIR / f"{state_name}_stack.npy"
        np.save(save_path, stack)
        stack_path = str(save_path)

    result_summary = {
        "state": state_name,
        "state_bbox": str(state_bbox),
        "roi_bbox": str(roi_bbox),
        "season_stats_path": str(season_stats_path),
        "stack_path": stack_path,
        "august_1_date": STATE_DATES[state_name]["august_1"],
        "september_1_date": STATE_DATES[state_name]["september_1"],
        "october_1_date": STATE_DATES[state_name]["october_1"],
        "end_of_season_date": STATE_DATES[state_name]["end_of_season"],
        "usable_forecast_dates": int((season_df["status"] == "usable").sum()),
        "failed_forecast_dates": int((season_df["status"] != "usable").sum()),
        "august_1_mean_evi": safe_scalar(season_df.loc[season_df["date_label"] == "august_1", "mean_evi"].iloc[0]),
        "september_1_mean_evi": safe_scalar(season_df.loc[season_df["date_label"] == "september_1", "mean_evi"].iloc[0]),
        "october_1_mean_evi": safe_scalar(season_df.loc[season_df["date_label"] == "october_1", "mean_evi"].iloc[0]),
        "end_of_season_mean_evi": safe_scalar(season_df.loc[season_df["date_label"] == "end_of_season", "mean_evi"].iloc[0]),
        "august_1_mean_ndvi": safe_scalar(season_df.loc[season_df["date_label"] == "august_1", "mean_ndvi"].iloc[0]),
        "september_1_mean_ndvi": safe_scalar(season_df.loc[season_df["date_label"] == "september_1", "mean_ndvi"].iloc[0]),
        "october_1_mean_ndvi": safe_scalar(season_df.loc[season_df["date_label"] == "october_1", "mean_ndvi"].iloc[0]),
        "end_of_season_mean_ndvi": safe_scalar(season_df.loc[season_df["date_label"] == "end_of_season", "mean_ndvi"].iloc[0]),
        "mean_valid_pixel_pct": round(float(season_df["valid_pixel_pct_after_clip"].mean()), 2),
        "mean_cloudy_pixel_pct": round(float(season_df["cloudy_pixel_pct_after_clip"].mean()), 2),
        "preview_paths": "; ".join(season_df["preview_path"].dropna().astype(str).tolist()),
        "yield_prediction_status": "NOT PREDICTED - HLS features/statistics created; Prithvi feature extraction attempted",
    }

    if RUN_PRITHVI and model is not None:
        try:
            print(f"  Model device: {next(model.parameters()).device}")
            print(f"  DEVICE variable: {DEVICE}")
            features, input_used = run_prithvi(model, stack)
            result_summary["prithvi_status"] = "success"
            result_summary["prithvi_input_used"] = input_used
            result_summary.update(summarize_features(features))
            print(f"  Prithvi succeeded using: {input_used}")
            del features
        except Exception as exc:
            result_summary["prithvi_status"] = "failed"
            result_summary["prithvi_error"] = str(exc)
            print(f"  WARNING: Prithvi failed, but HLS stats were saved: {exc}")

    del stack, chips
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    return result_summary


# ============================================================
# 11. Main
# ============================================================

def main():
    check_boundary_files()

    print("Logging into Earthdata...")
    earthaccess.login(persist=True)

    print("Configuring cloud COG access...")
    configure_cloud_cog_access()

    global model 
    model= BACKBONE_REGISTRY.build("prithvi_eo_v2_tiny_tl", pretrained=True)
    model = model.to(DEVICE)
    if RUN_PRITHVI:
        print("Loading Prithvi backbone...")
        
        # Force all internal buffers to DEVICE (fixes positional embedding device mismatch)
        for name, buf in model.named_buffers():
            buf.data = buf.data.to(DEVICE)
        model.eval()
        print("Prithvi loaded")

    final_results = []

    for state in PROCESS_STATES:
        try:
            summary = process_state(state, model)
            final_results.append(summary)
        except Exception as exc:
            print(f"ERROR for {state}: {exc}")
            final_results.append({
                "state": state,
                "error": str(exc),
                "yield_prediction_status": "FAILED BEFORE FEATURE EXTRACTION",
            })

        partial_path = OUTPUT_DIR / "hls_prithvi_partial_results.csv"
        pd.DataFrame(final_results).to_csv(partial_path, index=False)
        print(f"Saved partial summary: {partial_path}")

    final_df = pd.DataFrame(final_results)
    final_path = OUTPUT_DIR / "hls_prithvi_final_results.csv"
    final_df.to_csv(final_path, index=False)

    print("\nDone")
    print(f"Final summary saved to: {final_path}")
    print(f"Candidate diagnostics folder: {DIAGNOSTIC_DIR}")
    print(f"Preview images folder: {PREVIEW_DIR}")

    if "display" in globals():
        display(final_df)
    else:
        print(final_df)


if __name__ == "__main__":
    main()

print("=== PARAMETERS on CPU ===")
found_any = False

for name, param in model.named_parameters():
    if param.device.type == 'cpu':
        print(f"  PARAMETER on CPU: {name} | {param.device}")
        found_any = True
if not found_any:
    print("  None — all parameters are on GPU")

print("\n=== BUFFERS on CPU ===")
found_any = False
for name, buf in model.named_buffers():
    if buf.device.type == 'cpu':
        print(f"  BUFFER on CPU: {name} | {buf.device}")
        found_any = True
if not found_any:
    print("  None — all buffers are on GPU")

print("\n=== FULL ERROR (untruncated) ===")
import pandas as pd
df = pd.read_csv("data/outputs/hls_prithvi_final_results.csv")
for i, row in df.iterrows():
    print(f"\n{row['state']}:")
    print(row.get('prithvi_error', 'no error recorded'))


Running on: cuda
Logging into Earthdata...
Configuring cloud COG access...
Loading Prithvi backbone...
Prithvi loaded

Processing Colorado
State bbox: (-109.04188443282699, 37.017916855477324, -102.03307524801697, 41.02508928997431)
Small ROI bbox used for HLS search: (-105.59349759188976, 39.037615197341204, -105.50763195659721, 39.103284664624745)
Small ROI is about 6.72 km x 6.72 km
 Forecast point: august_1
  Specific date: 2025-08-01
  HLS search window: 2025-07-16T00:00:00 to 2025-08-17T23:59:59


/opt/conda/lib/python3.12/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


    Total granules found (HLSL30 + HLSS30): 23
  Metadata granules found for small ROI: 23
  Opening best candidates only: 12
    Candidate 1/12 | metadata cloud=5.0%
      skipped: too cloudy after ROI clip (91.4%)
    Candidate 2/12 | metadata cloud=13.0%
      usable | valid=38.1% | cloudy=61.9% | mean EVI=0.2423 | preview=data/preview_images/Colorado_august_1_rgb_preview.png
  Candidate diagnostics saved: data/diagnostics/Colorado_august_1_candidate_diagnostics.csv
 Forecast point: september_1
  Specific date: 2025-09-01
  HLS search window: 2025-08-16T00:00:00 to 2025-09-17T23:59:59


/opt/conda/lib/python3.12/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


    Total granules found (HLSL30 + HLSS30): 23
  Metadata granules found for small ROI: 23
  Opening best candidates only: 12
    Candidate 1/12 | metadata cloud=7.0%
      skipped: too cloudy after ROI clip (81.4%)
    Candidate 2/12 | metadata cloud=13.0%
      usable | valid=84.36% | cloudy=15.64% | mean EVI=0.1874 | preview=data/preview_images/Colorado_september_1_rgb_preview.png
  Candidate diagnostics saved: data/diagnostics/Colorado_september_1_candidate_diagnostics.csv
 Forecast point: october_1
  Specific date: 2025-10-01
  HLS search window: 2025-09-15T00:00:00 to 2025-10-17T23:59:59


...

/opt/conda/lib/python3.12/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


    Total granules found (HLSL30 + HLSS30): 16
  Metadata granules found for small ROI: 16
  Opening best candidates only: 12
    Candidate 1/12 | metadata cloud=0.0%
      usable | valid=93.77% | cloudy=6.23% | mean EVI=0.6163 | preview=data/preview_images/Missouri_august_1_rgb_preview.png
  Candidate diagnostics saved: data/diagnostics/Missouri_august_1_candidate_diagnostics.csv
 Forecast point: september_1
  Specific date: 2025-09-01
  HLS search window: 2025-08-16T00:00:00 to 2025-09-17T23:59:59


/opt/conda/lib/python3.12/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


    Total granules found (HLSL30 + HLSS30): 20
  Metadata granules found for small ROI: 20
  Opening best candidates only: 12
    Candidate 1/12 | metadata cloud=0.0%
      usable | valid=94.14% | cloudy=5.86% | mean EVI=0.4508 | preview=data/preview_images/Missouri_september_1_rgb_preview.png
  Candidate diagnostics saved: data/diagnostics/Missouri_september_1_candidate_diagnostics.csv
 Forecast point: october_1
  Specific date: 2025-10-01
  HLS search window: 2025-09-15T00:00:00 to 2025-10-17T23:59:59


KeyboardInterrupt: 

In [35]:
raw_df = pd.read_csv("data/CleanedCSV.csv")

TRAINING_YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

state_name_map = {
    "COLORADO": "Colorado",
    "IOWA": "Iowa",
    "MISSOURI": "Missouri",
    "NEBRASKA": "Nebraska",
    "WISCONSIN": "Wisconsin",
}

yield_df = (
    raw_df[
        (raw_df["GeoLvl"] == "STATE") &
        (raw_df["Year"].isin(TRAINING_YEARS))
    ]
    .copy()
    .rename(columns={
        "Year": "year",
        "State": "state",
        "Value(Bu/Acre)": "yield_bu_acre",
    })
)

yield_df["state"] = yield_df["state"].map(state_name_map)
yield_df = yield_df[["state", "year", "yield_bu_acre"]].reset_index(drop=True)

yield_df.to_csv("data/outputs/usda_historical_yields.csv", index=False)
print("Historical yields loaded from CSV:")
print(yield_df.to_string(index=False))

Historical yields loaded from CSV:
    state  year  yield_bu_acre
 Colorado  2023          122.0
     Iowa  2023          201.0
 Missouri  2023          153.0
 Nebraska  2023          182.0
Wisconsin  2023          176.0
 Colorado  2023          130.0
     Iowa  2023          203.0
 Missouri  2023          143.0
 Nebraska  2023          184.0
Wisconsin  2023          166.0
 Colorado  2023          124.0
     Iowa  2023          200.0
 Missouri  2023          147.0
 Nebraska  2023          173.0
Wisconsin  2023          171.0
 Colorado  2023          128.0
     Iowa  2023          199.0
 Missouri  2023          141.0
 Nebraska  2023          174.0
Wisconsin  2023          165.0
 Colorado  2023          130.0
     Iowa  2023          200.0
 Missouri  2023          145.0
 Nebraska  2023          177.0
Wisconsin  2023          165.0
 Colorado  2022          121.0
     Iowa  2022          200.0
 Missouri  2022          161.0
 Nebraska  2022          165.0
Wisconsin  2022          180.0
 Col

In [ ]:
class YieldHead(nn.Module):
    def __init__(self, feature_dim=192):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )
    def forward(self, features):
        pooled = features.mean(dim=1)
        return self.head(pooled).squeeze(-1)


def extract_features_for_year(state_name: str, year: int) -> Optional[np.ndarray]:
    print(f"\nExtracting features: {state_name} {year}")
    state_boundary = load_boundary(state_name)
    roi_wgs84 = make_small_roi_from_boundary(state_boundary, state_name)
    roi_bbox = get_bbox(roi_wgs84)

    dates = {
        "august_1":      f"{year}-08-01",
        "september_1":   f"{year}-09-01",
        "october_1":     f"{year}-10-01",
        "end_of_season": f"{year}-{END_OF_SEASON_DATES[state_name]}",
    }

    chips = []
    for date_label, forecast_date in dates.items():
        try:
            result = process_date_point(state_name, date_label, forecast_date, roi_wgs84, roi_bbox)
            chips.append(result["chip"])
        except Exception as e:
            print(f"  Skipping {date_label}: {e}")

    if not chips:
        print(f"  No chips found for {state_name} {year}")
        return None

    stack = build_temporal_stack(chips)
    model.to("cpu")
    stack_tensor = torch.from_numpy(stack.transpose(1, 0, 2, 3).copy()).unsqueeze(0).float()
    with torch.no_grad():
        features = model(stack_tensor)
    model.to(DEVICE)

    if isinstance(features, (list, tuple)):
        features = features[-1]
    return features.detach().float().cpu().numpy()


# Extract features for all training years
print("Extracting Prithvi features for training years...")
print("This will take 20-30 minutes...")

training_features = []
training_labels = []

for _, row in yield_df.iterrows():
    features_np = extract_features_for_year(row["state"], int(row["year"]))
    if features_np is not None:
        training_features.append(features_np)
        training_labels.append(row["yield_bu_acre"])
        print(f"  {row['state']} {row['year']} -> {row['yield_bu_acre']} bu/acre | shape: {features_np.shape}")

print(f"\nTotal training samples collected: {len(training_features)}")

In [34]:
if len(training_features) < 3:
    print("Not enough training samples. Need at least 3.")
else:
    X = torch.tensor(np.stack(training_features, axis=0)).float().squeeze(1)  # (N, 785, 192)
    y = torch.tensor(training_labels).float()  # (N,)

    print(f"Training input shape: {X.shape}")
    print(f"Training labels (bu/acre): {[round(v,1) for v in training_labels]}")

    # Normalize labels to help training converge
    y_mean = y.mean()
    y_std = y.std()
    y_norm = (y - y_mean) / y_std

    yield_head = YieldHead(feature_dim=X.shape[-1])
    optimizer = torch.optim.Adam(yield_head.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    yield_head.train()
    for epoch in range(300):
        optimizer.zero_grad()
        preds = yield_head(X)
        loss = loss_fn(preds, y_norm)
        loss.backward()
        optimizer.step()

        if epoch % 50 == 0:
            preds_real = preds.detach() * y_std + y_mean
            print(f"  Epoch {epoch:3d} | Loss: {loss.item():.4f} | "
                  f"Pred range: {preds_real.min():.1f} - {preds_real.max():.1f} bu/acre")

    # Save trained head
    torch.save({
        "state_dict": yield_head.state_dict(),
        "y_mean": y_mean.item(),
        "y_std": y_std.item(),
        "feature_dim": X.shape[-1],
    }, "data/outputs/yield_head_trained.pt")
    print("\nYieldHead saved to data/outputs/yield_head_trained.pt")

Training input shape: torch.Size([4, 785, 192])
Training labels (bu/acre): [122.0, 201.0, 153.0, 182.0]
  Epoch   0 | Loss: 0.7801 | Pred range: 161.9 - 165.3 bu/acre
  Epoch  50 | Loss: 0.6235 | Pred range: 163.8 - 172.9 bu/acre
  Epoch 100 | Loss: 0.9116 | Pred range: 142.8 - 184.3 bu/acre
  Epoch 150 | Loss: 0.3922 | Pred range: 150.5 - 185.9 bu/acre
  Epoch 200 | Loss: 0.0496 | Pred range: 131.5 - 195.9 bu/acre
  Epoch 250 | Loss: 0.0334 | Pred range: 131.6 - 198.8 bu/acre

YieldHead saved to data/outputs/yield_head_trained.pt


FileNotFoundError: [Errno 2] No such file or directory: 'data/outputs/yield_head_trained.pt'